In [ ]:
import pandas as pd
from runeq import initialize
# Pull your rune authentication yaml from ~/.rune/config (C:\Users\<USERNAME>>\.rune\config on Windows)
initialize()


In [ ]:
# To see information about your authenticated user
from runeq.resources.user import get_current_user 
my_user = get_current_user()
print(my_user) 
print('ActiveOrg:',my_user.active_org_name)

In [ ]:
# Get all the available patients and all the devices for each patient
from runeq.resources.patient import get_all_patients
patients = get_all_patients()
for patient in patients: 
    print(patient)
    for device in patient.devices: 
        print(' ',device)

In [ ]:
# Get the streams available for a particular patient
from runeq.resources.stream_metadata import get_patient_stream_metadata
patient_id = 'rcs07'
patient_streams = get_patient_stream_metadata(patient_id)
print(f'Found {len(patient_streams)} total streams')
stream_df = patient_streams.to_dataframe()
display(stream_df)

categories = stream_df['category'].unique().tolist()
print(f'Found the following categories of streams: \n  {categories}')

In [ ]:
# Look at the more specific attributes of the motion-related streams
grav_streams = patient_streams.filter(category='motion', measurement='gravity')
display(grav_streams.to_dataframe())

In [ ]:
# Make a function for more customizable filtering of the available streams
from runeq.resources.stream_metadata import get_stream_metadata, get_stream_availability_dataframe
stream_meta = get_stream_metadata(list(grav_streams.ids())[0])

def is_x_stream(stream):
    return 'axis' in stream.parameters.keys() and stream.parameters['axis'] in ['x'] # , 'y', 'z']


x_streams = patient_streams.filter(filter_function=is_x_stream, measurement='user')
display(x_streams.to_dataframe())

# Get available times for May 4, 2023
try:
    available = x_streams.get_batch_availability_dataframe(
        start_time=1681408661, end_time=1681427545, timezone=-28800,
        batch_operation='any', resolution=60*15)
except Exception as e:
    print(e)
else:
    display(available)



In [ ]:
import numpy as np
from runeq.resources.patient import get_device
# Get a list of devices with data available on a given day:
date = pd.Timestamp(1681427545, unit='s', tz='US/Pacific')
end_of_day = date + pd.Timedelta(hours=4)
resolution = 300

cumulative_avail = {}
for device in patient_streams.to_dataframe()['device_id'].unique():
    device_streams = x_streams.filter(device_id=device)
    try:
        device_avail = device_streams.get_batch_availability_dataframe(
            start_time=int(date.timestamp()), end_time=int(end_of_day.timestamp()), timezone=-28800,
            resolution=resolution, batch_operation='any'
        )
    except Exception as e:
        print(f'Device {device}')
        print(f'    {e}')
    else:
        # There is some data available for this device
        if sum(device_avail['availability']) > 0:
            print(f'Found data for {device}: {get_device(patient, device).name}')
            cumulative_avail[device] = device_avail
        else:
            print(f'No data for {device}')
   


In [ ]:
# Given the devices from above, find the range of times with available data
if len(cumulative_avail):
    devices = list(cumulative_avail.keys())
    raw_avail = [df['availability'].to_numpy() for df in cumulative_avail.values()]
    cumulative = [np.sum(raw_avail, axis=0) >= 1][0]
    avail_times = (cumulative_avail[devices[0]]['time'][cumulative]).to_numpy()
    start, end = int(pd.Timestamp(avail_times[0]).timestamp()), int(pd.Timestamp(avail_times[-1]).timestamp()) + resolution
    print(f'Data for {devices} available between {start} and {end}')
else:
    print('NO DATA FOUND IN THE GIVEN TIME RANGE!')

In [ ]:
# Get the data for the available time range
desired_ids = devices

all_data = []
for device in desired_ids:
    desired_streams = get_patient_stream_metadata('rcs07', device).filter(category='motion')
    print(f'Data for {device}: {get_device(patient, device).name}')
    print(f'Found {len(list(desired_streams.ids()))} streams over {(end - start) / 60} minutes.')
    for stream_id in desired_streams.ids():
        stream = get_stream_metadata(stream_id,)
        print(f'Fetching: {stream.parameters}')
        stream_data = stream.get_stream_dataframe(
            start_time=start, end_time=end
        )
        all_data.append(stream_data)

In [ ]:
desired_streams.to_dataframe()